### **Setup: Imports und Initialisierung**

In [ ]:
import sys
import os
import pandas as pd
from bson import json_util

sys.path.append(os.path.abspath('..'))

from quinex_application.mongos_store import MongoStore
from quinex_application.quinex_client import QuinexClient
from quinex_application.quinex_postprocessing import QuinexPostprocessing

In [ ]:
quinex = QuinexClient()
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2024_closed")
qpost = QuinexPostprocessing()

### **Load Analysis from MongoDB**

In [3]:
analysis = []
# "papers_2010_abstract_fallback"
# Jahre 2010–2025
for year in range(2010, 2026):
    analysis.append(f"papers_{year}")
    analysis.append(f"papers_{year}_closed")
    analysis.append(f"papers_{year}_abstract_fallback")

analysis += ["papers_wos", "papers_wos_closed", "papers_wos_abstract_fallback"]

meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")
df_path = "../data/quinex/df_quinex.parquet"
df_metadata_path = "../data/quinex/df_quinex_metadata.parquet"

for ana in analysis:
    a = mongodb.get_collection(ana)
    print(ana, len(a))
    
    if "_abstract_fallback" not in ana:
        df_quinex_meta = qpost.create_metadata_df(a, meta_df)
        qpost.save_df(df_quinex_meta, df_metadata_path)

    df_quinex = qpost.create_information_df(a, meta_df)
    qpost.save_df(df_quinex, df_path)


papers_2010 74
✅ New DataFrame saved to '../data/quinex/df_quinex_metadata.parquet'
⚠️  Maxed integer number achieved for DOI: 10.3390/su2123650
✅ New DataFrame saved to '../data/quinex/df_quinex.parquet'
papers_2010_closed 206
✅ Existing file found. Appended and saved updated DataFrame to '../data/quinex/df_quinex_metadata.parquet'
✅ Existing file found. Appended and saved updated DataFrame to '../data/quinex/df_quinex.parquet'
papers_2010_abstract_fallback 499
✅ Existing file found. Appended and saved updated DataFrame to '../data/quinex/df_quinex.parquet'
papers_2011 94
✅ Existing file found. Appended and saved updated DataFrame to '../data/quinex/df_quinex_metadata.parquet'
⚠️  Maxed integer number achieved for DOI: 10.1016/j.egypro.2011.05.075
✅ Existing file found. Appended and saved updated DataFrame to '../data/quinex/df_quinex.parquet'
papers_2011_closed 277
✅ Existing file found. Appended and saved updated DataFrame to '../data/quinex/df_quinex_metadata.parquet'
⚠️ Entferne 1

In [ ]:
analysis = mongodb.get_collection("papers_2011_closed")
len(analysis)

In [ ]:
# n = len(analysis)
# half = n // 2

# chunks = [
#     analysis[:half],
#     analysis[half:]
# ]

# for i, chunk in enumerate(chunks, start=1):
#     filename = f"papers_2025_closed_{i}.json"
#     with open(filename, "w", encoding="utf-8") as f:
#         f.write(json_util.dumps(chunk, indent=2, ensure_ascii=False))

with open("papers_2025_closed.json", "w", encoding="utf-8") as f:
    f.write(json_util.dumps(analysis, indent=2, ensure_ascii=False))

### **Load Analysis Metadata into DataFrame**

In [ ]:
meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")

df_quinex_meta = qpost.create_metadata_df(analysis, meta_df)

In [ ]:
df_path = "../data/quinex/df_quinex_metadata.parquet"
qpost.save_df(df_quinex_meta, df_path)

### **Load Analysis Results into DataFrame**

In [ ]:
meta_df = pd.read_parquet("../data/paper/paper_dataset.parquet")

df_quinex = qpost.create_information_df(analysis, meta_df)

In [ ]:
df = df_quinex
df = df[df["entity"].str.contains("power plant", case=False, na=False)]
df_upload = df.iloc[0:120]
display(df_upload)
#df_upload.to_excel("test_upload2.xlsx", index=False)

In [ ]:
bad = df_quinex[
    df_quinex['quantity_normalized']
    .astype(str)
    .str.contains("10203040506070809010")
]

display(bad)

In [ ]:
df_path = "../data/quinex/df_quinex.parquet"
qpost.save_df(df_quinex, df_path)

### **Metadata Results**

In [ ]:
# Alle Paper-IDs, die in analysis vorkommen (mit Analyse)
analysis_ids = {paper["_id"] for paper in analysis if paper.get("analysis_result")}

# Alle Paper-IDs, die tatsächlich im DataFrame gelandet sind
df_ids = set(df_quinex["paper_id"].unique())

# Fehlende Paper mit Analyse, aber ohne quantitative Aussage
missing = analysis_ids - df_ids

print(f"🧾 Total analyzed papers (with analysis_result): {len(analysis_ids)}")
print(f"✅ Papers in DataFrame: {len(df_ids)}")
print(f"❌ Papers with no valid quantitative statements: {len(missing)}")

# Optional: welche das sind
print("Missing paper_ids:", missing)

In [ ]:
# Gesamtanzahl der quantitativen Aussagen
total_statements = len(df_quinex)

# Durchschnittliche Anzahl an Aussagen pro Paper
avg_statements_per_paper = total_statements / len(df_ids)

print(f"📊 Total quantitative statements: {total_statements}")
print(f"➗ Average statements per paper: {avg_statements_per_paper:.2f}")

### **Apply Levenshtein Method with TechDB Hirarchy**

In [ ]:
df_quinex = qpost.load_df("../data/quinex/df_quinex.parquet")

In [ ]:
df_quinex.head(1)

In [ ]:
import requests
url = "http://134.94.117.230:8000/api/bulk-upload/map-mention/"

data = {
    "mention_type": "Entity",
    "mention": "battery"
}
response = requests.post(url, data=data)
print(response.json())

In [ ]:
df_quinex.rename(columns={
    'entity': 'Entity',
    'property': 'Property',
    'value_normalized': 'Value',
    'unit_normalized': 'Unit',
    #'quote': 'Quote',
}, inplace=True)

In [ ]:
import requests
import base64
from io import BytesIO

df = df_quinex.head(100)

excel_buffer = BytesIO()
df.to_excel(excel_buffer, index=False)
excel_buffer.seek(0)  # WICHTIG: Zeiger zurücksetzen auf Anfang

url = "http://134.94.117.230:8000/api/bulk-upload/upload-data/"

files = {
    "file": ("data.xlsx", excel_buffer, "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")
}
data = {
    "check_data": "true",   # oder "false"
    "upload_data": "false"  # oder "true"
}

# POST-Request
response = requests.post(url, files=files, data=data)
result = response.json()
print(result["message"])

file_bytes = base64.b64decode(result['file'])
excel_buffer = BytesIO(file_bytes)
df_mapped = pd.read_excel(excel_buffer)
display(df_mapped)